<a href="https://colab.research.google.com/github/mistyy12/flyrank-ML/blob/main/notebooks_finished/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mistyy12/flyrank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb huggingface_hub

In [4]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [5]:

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I prioritise pages with high search exposure but low click-through rate (CTR), using available GSC measuremenets from January-March 2026. CTR is total valid clicks divided by total impressions over total valid impressions.

The scoring rule and reason codes are:

- 2 points: At least 500 impressions and CTR below 1% (high_exposure_low_ctr).

- 1 extra point: Those pages also have an average search position between 1 and 10 (good_position_low_ctr).

- Unscored: Does not meet the minimum valid-history requirements (insufficient_history)

I use gsc_data_available to identify available measurements. Missing position receive no bonus. I rank pages by score descending, then by impressions descending to prioritise higher-exposure pages within same score.  

April measurements are reserved for outcome evaluation and do not influence the score. These thresholds are initial  assumptions. The ranking does not prove that a page needs refreshing or that an intervention will improve performance.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
if "con" not in globals():
    print("The DuckDB connection is missing. Run the setup cells first.")
else:
    display(con.sql("SHOW TABLES").df())

,name


In [7]:
# Point DuckDB to the January–March feature partitions.
REL = "hf://datasets/FlyRank/internship-warehouse"

feature_paths = [
    f"{REL}/fact_content_daily_performance/month={month}/*.parquet"
    for month in ["2026-01", "2026-02", "2026-03"]
]

# Create a view without loading all daily rows into pandas.
con.read_parquet(feature_paths).create_view(
    "contract_daily",
    replace=True
)

# Confirm that the view now exists.
display(con.sql("SHOW TABLES").df())

,name
0,contract_daily


In [8]:
#Aggregate daily data into one row per client-page
page_df = con.sql("""
    WITH daily AS (
        SELECT *,
            (
                gsc_data_available IS TRUE
                AND gsc_impressions IS NOT NULL
                AND gsc_clicks IS NOT NULL
            ) AS valid_metrics
        FROM contract_daily
        WHERE report_date BETWEEN DATE '2026-01-01'
                              AND DATE '2026-03-31'
    )
    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(DISTINCT report_date) FILTER (
            WHERE valid_metrics
        ) AS valid_gsc_days,

        SUM(gsc_impressions) FILTER (
            WHERE valid_metrics
        ) AS impressions,

        SUM(gsc_clicks) FILTER (
            WHERE valid_metrics
        ) AS clicks,

        SUM(gsc_sum_position) FILTER (
            WHERE valid_metrics
              AND gsc_impressions > 0
              AND gsc_sum_position IS NOT NULL
              AND gsc_avg_position > 0
        ) * 1.0 / NULLIF(
            SUM(gsc_impressions) FILTER (
                WHERE valid_metrics
                  AND gsc_impressions > 0
                  AND gsc_sum_position IS NOT NULL
                  AND gsc_avg_position > 0
            ), 0
        ) AS avg_position

    FROM daily
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Total client–pages: {len(page_df):,}")
display(page_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total client–pages: 349,411


,client_hash_id,content_hash_id,valid_gsc_days,impressions,clicks,avg_position
0,client_e547b89c05043229,content_4baa677479d16fb5,88,3345.0,11.0,13.947384
1,client_e547b89c05043229,content_6a167adf0f285e1c,88,4269.0,18.0,4.567112
2,client_e547b89c05043229,content_b2c8677f822e152a,88,14292.0,66.0,6.371537
3,client_e547b89c05043229,content_c6ac71d72dbccc86,88,1533.0,3.0,6.350831
4,client_e547b89c05043229,content_c4a9f203b6708ddf,88,6557.0,10.0,9.356413


In [9]:
#Calculate CTR and Eligibility

import pandas as pd

MIN_GSC_DAYS = 60
MIN_IMPRESSIONS = 500
LOW_CTR = 0.01  # 1%

page_df["ctr"] = (
    page_df["clicks"]
    / page_df["impressions"].where(page_df["impressions"] > 0)
)

page_df["eligible"] = (
    (page_df["valid_gsc_days"] >= MIN_GSC_DAYS)
    & page_df["ctr"].notna()
)

page_df["eligibility_note"] = "eligible"

page_df.loc[
    page_df["valid_gsc_days"] < MIN_GSC_DAYS,
    "eligibility_note"
] = "insufficient_history"

page_df.loc[
    (page_df["valid_gsc_days"] >= MIN_GSC_DAYS)
    & page_df["ctr"].isna(),
    "eligibility_note"
] = "undefined_ctr"

print(page_df["eligibility_note"].value_counts())

eligibility_note
insufficient_history    278979
eligible                 70432
Name: count, dtype: int64


In [10]:
# Apply the score, reason code and action
candidate = (
    page_df["eligible"]
    & (page_df["impressions"] >= MIN_IMPRESSIONS)
    & (page_df["ctr"] < LOW_CTR)
)

page_df["position_bonus"] = (
    candidate
    & page_df["avg_position"].between(1, 10)
)

# Unscored pages have a missing score, rather than zero.
page_df["score"] = pd.Series(
    pd.NA, index=page_df.index, dtype="Int64"
)

page_df.loc[page_df["eligible"], "score"] = 0
page_df.loc[candidate, "score"] = 2
page_df.loc[page_df["position_bonus"], "score"] = 3

page_df["reason_code"] = pd.Series(
    pd.NA, index=page_df.index, dtype="string"
)
page_df.loc[candidate, "reason_code"] = "high_exposure_low_ctr"

page_df["action"] = "monitor"
page_df.loc[candidate, "action"] = "refresh"
page_df.loc[~page_df["eligible"], "action"] = "check_data"

In [11]:
# Rank and display the queue -> rank through score, impressions, client hash, content hash
queue_df = page_df.sort_values(
    ["score", "impressions", "client_hash_id", "content_hash_id"],
    ascending=[False, False, True, True],
    na_position="last"
).reset_index(drop=True)

queue_df["rank"] = pd.Series(
    pd.NA, index=queue_df.index, dtype="Int64"
)

scored = queue_df["score"].notna()
queue_df.loc[scored, "rank"] = range(1, int(scored.sum()) + 1)

queue_df = queue_df[
    [
        "rank", "client_hash_id", "content_hash_id",
        "valid_gsc_days", "impressions", "clicks",
        "ctr", "avg_position", "eligible",
        "eligibility_note", "position_bonus",
        "score", "reason_code", "action"
    ]
]

display(queue_df.head(10))

,rank,client_hash_id,content_hash_id,valid_gsc_days,impressions,clicks,ctr,avg_position,eligible,eligibility_note,position_bonus,score,reason_code,action
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,88,830289.0,8122.0,0.009782,2.388430,True,eligible,True,3,high_exposure_low_ctr,refresh
1,2,client_73cda7b4e4f265ea,content_e241d6415ac9e534,90,509684.0,1255.0,0.002462,3.238771,True,eligible,True,3,high_exposure_low_ctr,refresh
2,3,client_e547b89c05043229,content_ec2e0346994fb5a5,88,509532.0,2515.0,0.004936,2.630747,True,eligible,True,3,high_exposure_low_ctr,refresh
3,4,client_62f4a7e64f5e0096,content_b99ea6861864dea5,90,463218.0,811.0,0.001751,4.148442,True,eligible,True,3,high_exposure_low_ctr,refresh
4,5,client_62f4a7e64f5e0096,content_acbcc847f8996314,90,433095.0,641.0,0.001480,3.810441,True,eligible,True,3,high_exposure_low_ctr,refresh
5,6,client_62f4a7e64f5e0096,content_f107e54b10b43725,82,427749.0,2235.0,0.005225,3.238792,True,eligible,True,3,high_exposure_low_ctr,refresh
6,7,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,90,407604.0,1614.0,0.003960,5.748651,True,eligible,True,3,high_exposure_low_ctr,refresh
7,8,client_73cda7b4e4f265ea,content_471d9cabce329a66,90,393421.0,972.0,0.002471,4.981224,True,eligible,True,3,high_exposure_low_ctr,refresh
8,9,client_73cda7b4e4f265ea,content_4d0d79fc12632ef8,90,388737.0,1981.0,0.005096,4.604980,True,eligible,True,3,high_exposure_low_ctr,refresh
9,10,client_e547b89c05043229,content_1e921148b5fee86a,88,384139.0,474.0,0.001234,5.590471,True,eligible,True,3,high_exposure_low_ctr,refresh


In [12]:
# Check and write the csv
from pathlib import Path

# One row per client–page.
assert not queue_df.duplicated(
    ["client_hash_id", "content_hash_id"]
).any()

# Ineligible pages must remain unscored.
assert queue_df.loc[
    ~queue_df["eligible"], "score"
].isna().all()

# Eligible scores must follow the rule.
assert queue_df.loc[
    queue_df["eligible"], "score"
].isin([0, 2, 3]).all()

selected = queue_df["score"].fillna(0) > 0
assert queue_df.loc[selected, "reason_code"].notna().all()
assert queue_df.loc[selected, "action"].eq("refresh").all()

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue_df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Total client–pages: {len(queue_df):,}")
print("\nScore counts:")
print(queue_df["score"].value_counts(dropna=False))

Saved: work/outputs/baseline_action_score.csv
Total client–pages: 349,411

Score counts:
score
<NA>    278979
3        34939
2        21297
0        14196
Name: count, dtype: Int64


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Two limitations:
1. About 80% of observed pages are unscored because they fail your history requirement.
2. Your top 20 come from only three clients. The global impressions tie-breaker can concentrate recommendations among higher-exposure clients.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue_df.loc[
    queue_df["score"].fillna(0) > 0,
    [
        "rank", "client_hash_id", "content_hash_id",
        "valid_gsc_days", "impressions", "clicks",
        "ctr", "avg_position", "score", "action", "reason_code"
    ]
].head(20).copy()

# Easier to read than a CTR fraction.
top20["ctr_pct"] = top20["ctr"] * 100

display(top20)


,rank,client_hash_id,content_hash_id,valid_gsc_days,impressions,clicks,ctr,avg_position,score,action,reason_code,ctr_pct
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,88,830289.0,8122.0,0.009782,2.388430,3,refresh,high_exposure_low_ctr,0.978214
1,2,client_73cda7b4e4f265ea,content_e241d6415ac9e534,90,509684.0,1255.0,0.002462,3.238771,3,refresh,high_exposure_low_ctr,0.246231
2,3,client_e547b89c05043229,content_ec2e0346994fb5a5,88,509532.0,2515.0,0.004936,2.630747,3,refresh,high_exposure_low_ctr,0.493590
3,4,client_62f4a7e64f5e0096,content_b99ea6861864dea5,90,463218.0,811.0,0.001751,4.148442,3,refresh,high_exposure_low_ctr,0.175080
4,5,client_62f4a7e64f5e0096,content_acbcc847f8996314,90,433095.0,641.0,0.001480,3.810441,3,refresh,high_exposure_low_ctr,0.148004
5,6,client_62f4a7e64f5e0096,content_f107e54b10b43725,82,427749.0,2235.0,0.005225,3.238792,3,refresh,high_exposure_low_ctr,0.522503
6,7,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,90,407604.0,1614.0,0.003960,5.748651,3,refresh,high_exposure_low_ctr,0.395973
7,8,client_73cda7b4e4f265ea,content_471d9cabce329a66,90,393421.0,972.0,0.002471,4.981224,3,refresh,high_exposure_low_ctr,0.247064
8,9,client_73cda7b4e4f265ea,content_4d0d79fc12632ef8,90,388737.0,1981.0,0.005096,4.604980,3,refresh,high_exposure_low_ctr,0.509599
9,10,client_e547b89c05043229,content_1e921148b5fee86a,88,384139.0,474.0,0.001234,5.590471,3,refresh,high_exposure_low_ctr,0.123393


In [15]:
# top 20, ge -> greater than or equal to

top20["review_confidence"] = "Moderate"
top20["confidence_note"] = (
    "Meets the rule with sufficient history; "
    "query-level context still needs review."
)

near_cutoff = top20["ctr"].ge(0.008)

unusual_metrics = (
    top20["impressions"].ge(100_000)
    & top20["clicks"].le(5)
    & top20["avg_position"].between(1, 3)
)

top20.loc[near_cutoff, "review_confidence"] = "Low"
top20.loc[near_cutoff, "confidence_note"] = (
    "CTR is close to the cutoff; selection is threshold-sensitive."
)
display(top20)

,rank,client_hash_id,content_hash_id,valid_gsc_days,impressions,clicks,ctr,avg_position,score,action,reason_code,ctr_pct,review_confidence,confidence_note
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,88,830289.0,8122.0,0.009782,2.388430,3,refresh,high_exposure_low_ctr,0.978214,Low,CTR is close to the cutoff; selection is thres...
1,2,client_73cda7b4e4f265ea,content_e241d6415ac9e534,90,509684.0,1255.0,0.002462,3.238771,3,refresh,high_exposure_low_ctr,0.246231,Moderate,Meets the rule with sufficient history; query-...
2,3,client_e547b89c05043229,content_ec2e0346994fb5a5,88,509532.0,2515.0,0.004936,2.630747,3,refresh,high_exposure_low_ctr,0.493590,Moderate,Meets the rule with sufficient history; query-...
3,4,client_62f4a7e64f5e0096,content_b99ea6861864dea5,90,463218.0,811.0,0.001751,4.148442,3,refresh,high_exposure_low_ctr,0.175080,Moderate,Meets the rule with sufficient history; query-...
4,5,client_62f4a7e64f5e0096,content_acbcc847f8996314,90,433095.0,641.0,0.001480,3.810441,3,refresh,high_exposure_low_ctr,0.148004,Moderate,Meets the rule with sufficient history; query-...
5,6,client_62f4a7e64f5e0096,content_f107e54b10b43725,82,427749.0,2235.0,0.005225,3.238792,3,refresh,high_exposure_low_ctr,0.522503,Moderate,Meets the rule with sufficient history; query-...
6,7,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,90,407604.0,1614.0,0.003960,5.748651,3,refresh,high_exposure_low_ctr,0.395973,Moderate,Meets the rule with sufficient history; query-...
7,8,client_73cda7b4e4f265ea,content_471d9cabce329a66,90,393421.0,972.0,0.002471,4.981224,3,refresh,high_exposure_low_ctr,0.247064,Moderate,Meets the rule with sufficient history; query-...
8,9,client_73cda7b4e4f265ea,content_4d0d79fc12632ef8,90,388737.0,1981.0,0.005096,4.604980,3,refresh,high_exposure_low_ctr,0.509599,Moderate,Meets the rule with sufficient history; query-...
9,10,client_e547b89c05043229,content_1e921148b5fee86a,88,384139.0,474.0,0.001234,5.590471,3,refresh,high_exposure_low_ctr,0.123393,Moderate,Meets the rule with sufficient history; query-...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Rank 1: Refresh recommendation is sensitive to the threshold as confidence is low. Its CTR is 0.978%, only slightly below the 1% cutoff. I consider it a weak pick because small change in CTR could change the selection. However, its high impressions mean it still deserves investigation. I would check query-level CTR and daily trends before recommending a refresh.

Rank 13: Investigate before refreshing. This page has 362658 impressions but only 3 clicks, giving a CTR of approximately 0.000827%, despite an average position of 1.22. This unusual combination could reflect search-result features, query intent or measurement problems. I would verify the measurements and inspect query-level performance before deciding whether a refresh is appropriate. These metric alone does not justify pruning.

Leakage check: The baseline uses only January–March 2026 measurements: valid GSC days, impressions, clicks, CTR, and average position. No April outcome data, outcome-derived labels, or existing FlyRank product flags are used as scoring inputs. Client and content IDs are used for grouping and deterministic tie-breaking only.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.